In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import (
    PROJECT_ROOT,
    DATA_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    WEATHER_DIR,
    SOIL_DIR,
    RAW_DIR,
    OUTPUT_DIR,
    PROCESSED_DATASET,
    WEATHER_FEATHER,
    MERGED_DATA_DIR,
    interim_csb_path,
)


# CSB Combined Data Analysis

This notebook performs data analysis and exploration on combined processed Crop Sequence Boundaries (CSB) data. It is designed to be a general template that combine CSB data from different time periods (e.g., 2009-2016, 2017-2024).

In [ ]:
# Import necessary libraries
#%pip install geopandas seaborn shapely numpy matplotlib pyarrow statsmodels scipy
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.wkt import loads
import numpy as np
import pyarrow
from statsmodels.tsa.stattools import acf, adfuller
from scipy import stats
import os

# Ensure plots display inline
%matplotlib inline

In [ ]:
# --- Configuration ---
# Define the files and their specific year ranges/labels
# Note: Ensure these paths exist. 
data_sources = [
    {
        "path": str(interim_csb_path("20082015")),
        "label": "2008-2015",
        "years": range(2008, 2016)  # Use full range
    },
    {
        "path": str(interim_csb_path("20092016")),
        "label": "2016-Bridge",
        "years": [2016]             # ONLY take 2016 from this file to fill the gap
    },
    {
        "path": str(interim_csb_path("20172024")),
        "label": "2017-2024",
        "years": range(2017, 2025)  # Use full range
    }
]

# Mapping for Crop Codes (Shared across all files)
cdl_mapping = {
    1: 'Corn', 5: 'Soybeans', 36: 'Alfalfa', 37: 'Other Hay', 
    176: 'Grass/Pasture', 59: 'Sod/Grass Seed', 61: 'Fallow', 
    24: 'Winter Wheat', 121: 'Developed', 141: 'Forest',
    4: 'Sorghum', 21: 'Barley', 23: 'Spring Wheat'
}

def get_crop_name(code):
    return cdl_mapping.get(code, f"Other ({code})")

## 1. Robust Data Loading & Aggregation

We load multiple feather files, extract stats for specific years to avoid overlap, and combine them into a master dataset.

In [ ]:
combined_stats = []
combined_rotations = []
combined_geoms = []

for source in data_sources:
    path = source['path']
    print(f"Processing {source['label']} from {path}...")
    
    try:
        # Load the file
        df = pd.read_feather(path)
        
        # 1. EXTRACT ACREAGE TRENDS (The "Long" Format)
        # Find CDL columns
        cdl_cols = [c for c in df.columns if c.startswith('CDL')]
        
        # Melt: Convert wide (one row per field) to long (one row per field-year)
        melted = df.melt(
            id_vars=['CSBACRES', 'CNTY', 'CSBID'], 
            value_vars=cdl_cols, 
            var_name='Year_Raw', 
            value_name='Crop_Code'
        )
        
        # Clean Year column (remove 'CDL')
        melted['Year'] = melted['Year_Raw'].str.replace('CDL', '').astype(int)
        
        # FILTER: Only keep the years we specifically requested for this file
        target_years = list(source['years'])
        melted = melted[melted['Year'].isin(target_years)]
        
        # Aggregate: Sum acres by Year, County, and Crop
        # This removes the dependency on specific field IDs
        agg_df = melted.groupby(['Year', 'CNTY', 'Crop_Code'])['CSBACRES'].sum().reset_index()
        agg_df['Source_Window'] = source['label']
        combined_stats.append(agg_df)
        
        # 2. EXTRACT ROTATION STRATEGIES (Within-Window)
        # We calculate rotation type HERE, inside the loop, because the geometry is stable here.
        
        # Helper to classify rotation
        def classify_row_rotation(row_values):
            unique = set(row_values)
            unique_names = sorted([get_crop_name(c) for c in unique])
            if len(unique) == 1: return f"Cont. {unique_names[0]}"
            if len(unique) == 2: return f"Rot {'-'.join(unique_names)}"
            return "Complex/Mixed"

        # Apply classification (Optimized for speed)
        # Grab just the crop columns as a numpy array
        crop_matrix = df[cdl_cols].values
        rotation_types = [classify_row_rotation(row) for row in crop_matrix]
        
        # Count acres per rotation type
        rot_counts = df[['CSBACRES']].copy()
        rot_counts['Rotation_Type'] = rotation_types
        rot_agg = rot_counts.groupby('Rotation_Type')['CSBACRES'].sum().reset_index()
        rot_agg['Source_Window'] = source['label']
        combined_rotations.append(rot_agg)

        # 3. EXTRACT GEOMETRY (Append)
        # Keep only necessary columns to save memory
        geom_cols = ['CSBID', 'geometry', 'CNTY', 'CNTYFIPS']
        available_cols = [c for c in geom_cols if c in df.columns]
        geom_subset = df[available_cols].copy()
        geom_subset['Source_Window'] = source['label']
        combined_geoms.append(geom_subset)

    except Exception as e:
        print(f"  Error loading {path}: {e}")

# --- 2. Combine Results ---

if combined_stats:
    # Master Acreage DataFrame (2008-2024)
    master_stats_df = pd.concat(combined_stats, ignore_index=True)
    master_stats_df['Crop_Name'] = master_stats_df['Crop_Code'].map(get_crop_name)
    
    # Master Rotation DataFrame
    master_rot_df = pd.concat(combined_rotations, ignore_index=True)
    
    # Master Geometry DataFrame
    if combined_geoms:
        master_geom_df = pd.concat(combined_geoms, ignore_index=True)

    print("\nData Combination Complete!")
    print(f"Total Aggregated Records: {len(master_stats_df)}")
    print(f"Years covered: {master_stats_df['Year'].min()} to {master_stats_df['Year'].max()}")
    if 'master_geom_df' in locals():
        print(f"Total Geometry Records: {len(master_geom_df)}")
    
    display(master_stats_df.head())
else:
    print("No data loaded.")

In [ ]:
# Save Data

# Define base directories
base_output_dir = os.path.join("data", "merged_data")
crop_output_dir = os.path.join(base_output_dir, "merged_CSB_by_crop")
other_crop_output_dir = os.path.join(crop_output_dir, "merged_CSB_other")

# Create directories
os.makedirs(base_output_dir, exist_ok=True)
os.makedirs(crop_output_dir, exist_ok=True)
os.makedirs(other_crop_output_dir, exist_ok=True)

if 'master_stats_df' in locals():
    # 1. Store overall combined data
    total_path = os.path.join(base_output_dir, "merged_CSB_total.csv")
    master_stats_df.to_csv(total_path, index=False)
    print(f"Saved overall combined data to {total_path}")

    # 2. Filter by crop type and store annual data
    unique_crops = master_stats_df['Crop_Name'].unique()
    other_crops_dfs = []
    
    for crop in unique_crops:
        crop_df = master_stats_df[master_stats_df['Crop_Name'] == crop]
        
        # Check if it is an 'Other' crop
        if crop.startswith("Other ("):
            # Collect for aggregation
            other_crops_dfs.append(crop_df)
            
            # Save individual 'Other' crop to the specific subdirectory
            safe_crop_name = crop.replace('/', '_').replace(' ', '_').replace('(', '').replace(')', '')
            crop_path = os.path.join(other_crop_output_dir, f"merged_CSB_{safe_crop_name}.csv")
            crop_df.to_csv(crop_path, index=False)
            # print(f"Saved {crop} data to {crop_path}") # Optional: comment out to reduce noise
            
        else:
            # Save major crops to the main crop directory
            safe_crop_name = crop.replace('/', '_').replace(' ', '_')
            crop_path = os.path.join(crop_output_dir, f"merged_CSB_{safe_crop_name}.csv")
            crop_df.to_csv(crop_path, index=False)
            print(f"Saved {crop} data to {crop_path}")

    # 3. Save Aggregated 'Other' Crops
    if other_crops_dfs:
        combined_other_df = pd.concat(other_crops_dfs, ignore_index=True)
        combined_other_path = os.path.join(crop_output_dir, "merged_CSB_Other_Combined.csv")
        combined_other_df.to_csv(combined_other_path, index=False)
        print(f"Saved Aggregated Other Crops to {combined_other_path}")

if 'master_geom_df' in locals():
    # 4. Store geometry data
    # Check if geometry is WKB (bytes)
    if isinstance(master_geom_df['geometry'].iloc[0], bytes):
        from shapely import wkb
        print("Converting WKB to Geometry...")
        # Use gpd.GeoSeries.from_wkb if available (faster)
        geometry = gpd.GeoSeries.from_wkb(master_geom_df['geometry'])
        gdf = gpd.GeoDataFrame(master_geom_df, geometry=geometry)
        
        poly_path = os.path.join(base_output_dir, "merged_CSB_polygon.parquet")
        gdf.to_parquet(poly_path)
        print(f"Saved geometry data to {poly_path}")
    else:
        # Already geometry objects?
        pass


In [ ]:
# Data Exploration

if 'master_stats_df' in locals():
    # Inspect the first few rows to understand the data structure
    print("\nFirst 5 rows:")
    display(master_stats_df.head())

    # Inspect the last few rows to understand the data structure
    print("\nLast 5 rows:")
    display(master_stats_df.tail())

    # Inspect column data types and non-null counts
    print("\nDataFrame Info:")
    master_stats_df.info()

    # Summary statistics for numerical columns
    print("\nStatistical Description:")
    display(master_stats_df.describe())

### 4.1. Group & Classify Crop Sequences

In [ ]:
if 'master_rot_df' in locals():
    # Compare rotation strategies between the time windows
    plt.figure(figsize=(12, 6))
    top_rotations = master_rot_df.groupby('Rotation_Type')['CSBACRES'].sum().nlargest(10).index
    filtered_rot = master_rot_df[master_rot_df['Rotation_Type'].isin(top_rotations)]

    sns.barplot(data=filtered_rot, x='Rotation_Type', y='CSBACRES', hue='Source_Window')
    plt.title('Dominant Rotation Strategies by Time Period')
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Total Acres')
    plt.show()

### 4.2. Time Series Analysis & Transformations

We analyze the stability of crop acreage over time using ACF and ADF tests. We also apply transformations (Log, Differencing, Box-Cox) to stabilize variance and trends.

In [ ]:
if 'master_stats_df' in locals():
    # 1. Prepare Time Series Data (Sum across counties for state-level analysis)
    ny_trends = master_stats_df.groupby(['Year', 'Crop_Name'])['CSBACRES'].sum().reset_index()

    # Filter for top 5 crops for detailed analysis
    top_crops = ny_trends.groupby('Crop_Name')['CSBACRES'].sum().nlargest(5).index
    
    # Pivot to get years as index and crops as columns
    ts_pivot = ny_trends[ny_trends['Crop_Name'].isin(top_crops)].pivot(index='Year', columns='Crop_Name', values='CSBACRES')
    ts_pivot = ts_pivot.fillna(0) # Handle missing years if any
    
    # --- Helper Function for Stats ---
    def calculate_ts_stats(series, name):
        # ACF Lag 1
        acf_val = acf(series, nlags=1)[1] if len(series) > 1 else np.nan
        # ADF Test
        try:
            adf_res = adfuller(series.dropna())
            adf_stat, p_val = adf_res[0], adf_res[1]
            is_stationary = p_val < 0.05
        except:
            adf_stat, p_val, is_stationary = np.nan, np.nan, "Error"
        return {
            'Transformation': name,
            'ACF (Lag 1)': round(acf_val, 3),
            'ADF Stat': round(adf_stat, 3),
            'P-Value': round(p_val, 4),
            'Stationary?': is_stationary
        }

    # --- Analyze Each Crop with Transformations ---
    for crop in top_crops:
        print(f"\n--- Analysis for: {crop} ---")
        series = ts_pivot[crop]
        results = []

        # 1. Original Data
        results.append(calculate_ts_stats(series, 'Original'))

        # 2. Log Transformation (stabilize variance)
        # Add small constant to avoid log(0)
        log_series = np.log1p(series)
        results.append(calculate_ts_stats(log_series, 'Log (np.log1p)'))

        # 3. First Differencing (remove trend)
        diff_series = series.diff().dropna()
        results.append(calculate_ts_stats(diff_series, 'First Difference'))

        # 4. Box-Cox Transformation (power transform)
        # Box-Cox requires positive data
        if (series > 0).all():
            boxcox_series, lmbda = stats.boxcox(series)
            boxcox_series = pd.Series(boxcox_series, index=series.index)
            stats_res = calculate_ts_stats(boxcox_series, f'Box-Cox (lambda={lmbda:.2f})')
            results.append(stats_res)
        else:
            results.append({'Transformation': 'Box-Cox', 'Stationary?': 'Skipped (values <= 0)'})

        # 5. Log Differencing (Log + First Difference)
        log_diff_series = log_series.diff().dropna()
        results.append(calculate_ts_stats(log_diff_series, 'Log Difference'))


        # Display Results Table
        res_df = pd.DataFrame(results)
        display(res_df)

        # Plotting Comparisons
        fig, axes = plt.subplots(1, 4, figsize=(24, 4))
        
        # Plot Original
        axes[0].plot(series, marker='o', color='blue')
        axes[0].set_title(f'{crop} - Original')
        
        # Plot Log
        axes[1].plot(log_series, marker='o', color='green')
        axes[1].set_title(f'{crop} - Log Transformed')
        
        # Plot Differenced
        axes[2].plot(diff_series, marker='o', color='red')
        axes[2].set_title(f'{crop} - First Difference')
        
        # Plot Log Differenced
        axes[3].plot(log_diff_series, marker='o', color='purple')
        axes[3].set_title(f'{crop} - Log Difference')
        
        plt.tight_layout()
        plt.show()

### 5. GARCH volatility modeling (on log-differenced acreage)

This section models time-varying volatility in the (log) changes of acreage using a GARCH model.

#### Data used
For each crop, we use the **log-differenced** series

$$r_t = \Delta \log(1 + y_t) = \log(1+y_t) - \log(1+y_{t-1}),$$

where $y_t$ is total acres in year $t$ (aggregated statewide). The $\log(1+\cdot)$ transform avoids issues at 0.

#### Model form (GARCH(1,1))
We assume a mean equation and a conditional variance equation:

$$r_t = \mu + \varepsilon_t, \qquad \varepsilon_t = \sigma_t z_t,$$

$$\sigma_t^2 = \omega + \alpha\,\varepsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2,$$

with constraints $\omega > 0$, $\alpha \ge 0$, $\beta \ge 0$, and typically $\alpha + \beta < 1$ for covariance stationarity.

- $z_t$ is i.i.d. with mean 0 and variance 1 (often **Gaussian** or **Student-t**).

#### Fit evaluation metrics
We report standard likelihood-based metrics:

- **Log-likelihood**: $\ell(\hat\theta)$
- **AIC**: $\mathrm{AIC} = 2k - 2\ell(\hat\theta)$
- **BIC**: $\mathrm{BIC} = k\log(n) - 2\ell(\hat\theta)$

where $k$ is the number of estimated parameters and $n$ is the sample size.

#### Diagnostics (residual checks)
After fitting, we examine standardized residuals $\hat z_t = \hat\varepsilon_t/\hat\sigma_t$ and run Ljung–Box tests on:

- $\hat z_t$ (remaining autocorrelation)
- $\hat z_t^2$ (remaining ARCH effects)

A well-specified model should show little remaining serial correlation and little remaining autocorrelation in squared standardized residuals.

> Note: yearly series (e.g., 2008–2024) can be short for GARCH; results should be interpreted cautiously. If sample size is too small, the code below skips the fit.

In [ ]:
# --- GARCH fitting code ---
import warnings
warnings.filterwarnings('ignore')

from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf

try:
    from arch import arch_model
except ImportError as e:
    arch_model = None
    print("Missing dependency: 'arch'. Install it with: %pip install arch")


def fit_garch_11(r, dist='t'):
    """Fit a GARCH(1,1) with constant mean to a return series r (pd.Series)."""
    am = arch_model(r, mean='Constant', vol='GARCH', p=1, q=1, dist=dist, rescale=False)
    res = am.fit(disp='off', show_warning=False)
    return res


if 'ts_pivot' in locals() and 'top_crops' in locals() and arch_model is not None:
    garch_rows = []

    for crop in top_crops:
        # Use log-differences of log1p(acres) as "returns"
        y = ts_pivot[crop].astype(float)
        r = np.log1p(y).diff().dropna() * 100.0  # scale for numerical stability

        if len(r) < 12:
            print(f"Skipping {crop}: too few observations for stable GARCH fit (n={len(r)}).")
            continue

        print(f"\n--- GARCH(1,1) fit for: {crop} (n={len(r)}) ---")

        # Fit a Student-t GARCH(1,1)
        res = fit_garch_11(r, dist='t')

        # Fit metrics
        garch_rows.append({
            'Crop': crop,
            'n': int(len(r)),
            'Dist': 't',
            'LogLik': float(res.loglikelihood),
            'AIC': float(res.aic),
            'BIC': float(res.bic),
            'omega': float(res.params.get('omega', np.nan)),
            'alpha1': float(res.params.get('alpha[1]', np.nan)),
            'beta1': float(res.params.get('beta[1]', np.nan)),
            'mu': float(res.params.get('mu', np.nan)),
        })

        display(res.summary())

        # Diagnostics: standardized residuals
        std_resid = pd.Series(res.std_resid, index=r.index).dropna()

        lb_resid = acorr_ljungbox(std_resid, lags=[1, 2, 3], return_df=True)
        lb_sq = acorr_ljungbox(std_resid**2, lags=[1, 2, 3], return_df=True)

        diag_df = pd.DataFrame({
            'LB(resid) p-value': lb_resid['lb_pvalue'],
            'LB(resid^2) p-value': lb_sq['lb_pvalue'],
        })
        diag_df.index = [f"lag {i}" for i in diag_df.index]
        display(diag_df)

        # Plots
        fig, axes = plt.subplots(2, 2, figsize=(12, 7))

        axes[0, 0].plot(r, marker='o')
        axes[0, 0].set_title(f"{crop}: r_t = Δlog(1+acres) (×100)")

        cond_vol = pd.Series(res.conditional_volatility, index=r.index)
        axes[0, 1].plot(cond_vol, color='black')
        axes[0, 1].set_title(f"{crop}: Conditional volatility σ_t")

        axes[1, 0].plot(std_resid, color='tab:blue')
        axes[1, 0].axhline(0, color='gray', linewidth=1)
        axes[1, 0].set_title(f"{crop}: Standardized residuals")

        plot_acf(std_resid**2, lags=min(10, len(std_resid) - 1), ax=axes[1, 1])
        axes[1, 1].set_title(f"{crop}: ACF of standardized residuals²")

        plt.tight_layout()
        plt.show()

    if garch_rows:
        garch_fit_df = pd.DataFrame(garch_rows).sort_values(['AIC'])
        print("\n=== GARCH fit metrics (lower AIC/BIC is better) ===")
        display(garch_fit_df)
else:
    if 'ts_pivot' not in locals() or 'top_crops' not in locals():
        print("Run the time series transformation section first to create ts_pivot/top_crops.")
    if arch_model is None:
        print("Cannot fit GARCH because 'arch' is not installed in this environment.")